# De Text Mining a Representaciones Semánticas: Un Puente Conceptual

**Tecnicatura Superior en Ciencias de Datos e Inteligencia Artificial**  
**Procesamiento de Lenguaje Natural**

---

## Objetivo

En los cuadernos anteriores aprendiste a convertir texto en números usando Bag of Words y TF-IDF. Esas herramientas te permitieron construir matrices de frecuencia, detectar términos distintivos y comparar documentos. Fue un avance importante: pasaste del texto crudo a una representación estructurada.

Pero esas representaciones tienen limitaciones profundas que todavía no exploramos. Hoy vamos a hacerlas visibles.

En este cuaderno vas a:
- identificar y analizar las limitaciones semánticas de BoW y TF-IDF;
- descubrir por qué la similitud semántica es crucial para el PLN moderno;
- experimentar con vectores semánticos densos usando spaCy;
- preparar el terreno conceptual para word embeddings (Word2Vec, FastText, GloVe).

## Resultados de aprendizaje

Al finalizar este cuaderno vas a poder:
- explicar con ejemplos concretos por qué BoW y TF-IDF no capturan significado;
- usar spaCy para calcular similitud semántica entre palabras y textos;
- comparar los resultados de representaciones sparse (BoW) con representaciones densas (vectores semánticos);
- argumentar por qué los word embeddings representaron un cambio de paradigma en PLN.

## Relación con cuadernos anteriores

- En `005` construiste matrices BoW y TF-IDF con `CountVectorizer` y `TfidfVectorizer`.
- En `006` aplicaste esas técnicas a un corpus real de artículos periodísticos.
- En `003` trabajaste con spaCy para tokenizar, lematizar y analizar texto.
- **Acá** vas a confrontar las limitaciones de BoW/TF-IDF y descubrir una forma diferente de representar el significado de las palabras.

---

## 1. Configuración del entorno

Importamos las herramientas que ya conocés de cuadernos anteriores: `CountVectorizer` y `TfidfVectorizer` para las representaciones sparse, y las librerías de visualización.

In [1]:
# --- Importaciones ---

# pandas: tablas legibles para inspeccionar resultados.
import pandas as pd

# numpy: operaciones numéricas sobre vectores y matrices.
import numpy as np

# CountVectorizer: construye matrices Bag of Words (ya lo usamos en 005).
# TfidfVectorizer: construye matrices TF-IDF (ya lo usamos en 005 y 006).
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# cosine_similarity: mide el ángulo entre dos vectores (cuanto más cercano a 1, más similares).
from sklearn.metrics.pairwise import cosine_similarity

# matplotlib y seaborn: visualización.
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de advertencias y visualización
import warnings
warnings.filterwarnings('ignore')

plt.style.use('default')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Entorno configurado.")

ModuleNotFoundError: No module named 'sklearn'

---

## 2. Corpus de trabajo: fragmentos temáticos

Para los experimentos de hoy necesitamos un corpus pequeño y controlado donde podamos observar relaciones semánticas claras. Vamos a usar ocho fragmentos breves organizados en cuatro categorías temáticas: paternidad, fútbol, proyecto editorial y escritura.

Los fragmentos están inspirados en el estilo y las temáticas de Hernán Casciari, un escritor argentino que aborda estos temas recurrentemente en sus textos. No son citas textuales sino recreaciones que nos sirven como material de laboratorio.

In [ ]:
# --- Corpus de fragmentos temáticos ---
# Ocho textos breves organizados en cuatro categorías.
# Los usamos como laboratorio para explorar limitaciones y capacidades semánticas.

textos_casciari_inspirados = {
    "paternidad_temprana": """La nena se despertó a las tres de la mañana llorando.
    No sabía qué hacer, así que la alcé y caminé por la casa hasta que se calmó.
    Ser padre es lo más difícil que me ha tocado hacer en la vida.""",

    "paternidad_madura": """Mi hija se levantó temprano y me preparó el desayuno.
    Ya no es una bebé que llora por las noches, ahora es una persona
    que tiene sus propias opiniones sobre todo.""",

    "futbol_pasion": """El partido fue increíble. San Lorenzo jugó como nunca
    y la hinchada cantó durante los noventa minutos.
    El fútbol argentino tiene esa magia que no encuentro en ningún otro lugar.""",

    "futbol_nostalgia": """Viendo el encuentro por televisión recordé cuando iba
    a la cancha con mi viejo. La pasión azulgrana corría por nuestras venas
    y cada gol era una fiesta familiar.""",

    "orsai_nacimiento": """La idea de la revista surgió casi por casualidad.
    Quería hacer algo diferente, contar historias que no encontraba
    en otros medios. Los lectores respondieron de manera sorprendente.""",

    "orsai_consolidacion": """La publicación ya tiene su identidad propia.
    Los escritores que colaboran entienden el espíritu del proyecto
    y cada número es mejor que el anterior.""",

    "escritura_oficio": """Escribir es mi trabajo, pero también mi obsesión.
    Cada texto es una oportunidad de conectar con los lectores
    de una manera auténtica y directa.""",

    "escritura_arte": """Las palabras son mi herramienta para crear mundos.
    Cada relato es una invitación a los lectores para que
    construyan sus propias interpretaciones."""
}

# Convertimos a DataFrame para facilitar la manipulación
categorias = list(textos_casciari_inspirados.keys())
textos = list(textos_casciari_inspirados.values())

df_ejemplos = pd.DataFrame({
    "categoria": categorias,
    "texto": textos
})

cantidad_fragmentos = len(df_ejemplos)
print(f"Preparados {cantidad_fragmentos} fragmentos temáticos para análisis")
print("\nCategorías disponibles:")
for i in range(len(categorias)):
    categoria = categorias[i]
    print(f"  - {categoria}")


---

## 3. Las limitaciones de BoW y TF-IDF

En `005` ya viste que BoW pierde el orden de las palabras y que TF-IDF mejora la ponderación. Pero hay problemas más profundos que esas técnicas no resuelven. Vamos a hacerlos visibles con ejemplos concretos.

### Problema 1: La ceguera semántica

BoW trata cada palabra como una entidad completamente independiente. **"Padre" y "papá" son tan diferentes entre sí como "padre" y "computadora".**

Eso significa que si dos frases dicen lo mismo pero con palabras distintas, BoW las ve como textos completamente diferentes.

In [ ]:
# --- Ejemplo: sinónimos que BoW trata como palabras independientes ---

ejemplos_sinonimos = [
    "Mi papá me enseñó a jugar al fútbol cuando era chico",
    "Mi padre me mostró cómo patear la pelota de niño",
    "El viejo me explicó las reglas del juego siendo pequeño"
]

# Vectorizamos con BoW
vectorizador_bow = CountVectorizer()
matriz_bow = vectorizador_bow.fit_transform(ejemplos_sinonimos)
nombres_columnas = vectorizador_bow.get_feature_names_out()

# Creamos la tabla para visualizar
tabla_bow = pd.DataFrame(
    matriz_bow.toarray(),
    columns=nombres_columnas
)
tabla_bow.index = ["Frase 1", "Frase 2", "Frase 3"]

print("REPRESENTACIÓN BoW DE TRES FRASES SEMÁNTICAMENTE SIMILARES")
print("=" * 60)
print(tabla_bow)

# Calculamos la similitud coseno entre las frases
similitudes = cosine_similarity(matriz_bow)

sim_1_2 = similitudes[0, 1]
sim_1_3 = similitudes[0, 2]
sim_2_3 = similitudes[1, 2]

print("\nSIMILITUD COSENO ENTRE FRASES (según BoW):")
print(f"Frase 1 vs Frase 2: {sim_1_2:.3f}")
print(f"Frase 1 vs Frase 3: {sim_1_3:.3f}")
print(f"Frase 2 vs Frase 3: {sim_2_3:.3f}")


Observá los resultados: las tres frases hablan del mismo recuerdo (el padre enseñando fútbol al hijo), pero BoW las ve como textos casi completamente diferentes porque no comparten palabras exactas. La similitud es muy baja a pesar de que el significado es prácticamente idéntico.

Este es un problema fundamental, no un defecto de implementación.

### Problema 2: Pérdida del contexto semántico (polisemia)

La palabra "tiempo" puede significar cosas muy distintas según el contexto. BoW no distingue entre esos usos: para el modelo, son simplemente ocurrencias de la misma palabra.

In [ ]:
# --- Ejemplo de polisemia: una palabra, múltiples significados ---

frases_tiempo = [
    "No tengo tiempo para escribir hoy, estoy muy ocupado",
    "El tiempo vuela cuando estoy con mi familia",
    "En tiempo de pandemia cambió nuestra forma de vivir",
    "Hace tiempo que no veo una película tan buena",
    "El tiempo atmosférico está muy cambiante esta semana"
]

vectorizador_tiempo = CountVectorizer()
matriz_tiempo = vectorizador_tiempo.fit_transform(frases_tiempo)

# Buscamos la columna correspondiente a "tiempo"
nombres_tiempo = vectorizador_tiempo.get_feature_names_out()
indice_tiempo = np.where(nombres_tiempo == 'tiempo')[0][0]

print("ANÁLISIS DE POLISEMIA: LA PALABRA 'TIEMPO'")
print("=" * 45)
print("\nFrecuencia de 'tiempo' en cada frase:")

for i in range(len(frases_tiempo)):
    frase = frases_tiempo[i]
    frecuencia = matriz_tiempo[i, indice_tiempo]
    fragmento = frase[:50]
    print(f"Frase {i + 1}: {frecuencia} ocurrencia(s)")
    print(f"  Contexto: {fragmento}...")

print("\nREFLEXIÓN:")
print("BoW cuenta 5 ocurrencias de 'tiempo', pero cada una tiene un significado diferente:")
print("  - Tiempo como recurso (frase 1)")
print("  - Tiempo como experiencia subjetiva (frase 2)")
print("  - Tiempo como período histórico (frase 3)")
print("  - Tiempo como duración (frase 4)")
print("  - Tiempo como condición climática (frase 5)")


BoW/TF-IDF no puede distinguir entre estos usos semánticamente diferentes. Para estas técnicas, "tiempo" es siempre la misma palabra sin importar el contexto.

### Problema 3: Incapacidad para capturar relaciones conceptuales

Palabras como "hinchada", "cancha" y "gol" forman un campo semántico relacionado con el fútbol. Pero BoW/TF-IDF no puede saber eso: para estas técnicas, la relación entre "hinchada" y "cancha" es la misma que entre "hinchada" y "computadora".

In [ ]:
# --- Campos semánticos que TF-IDF no puede relacionar ---

campos_semanticos = {
    "Fútbol (explícito)": "El partido de fútbol fue emocionante, San Lorenzo ganó",
    "Fútbol (implícito 1)": "La hinchada cantó durante todo el encuentro azulgrana",
    "Fútbol (implícito 2)": "El Ciclón mostró gran juego y la tribuna se volvió loca",
    "Escritura (explícito)": "Escribir es mi pasión, cada texto es una aventura",
    "Escritura (implícito 1)": "La narración fluía y las palabras cobraban vida",
    "Escritura (implícito 2)": "El relato cautivó a los lectores desde el primer párrafo"
}

textos_campos = list(campos_semanticos.values())
labels_campos = list(campos_semanticos.keys())

# Vectorización con TF-IDF
vectorizador_campos = TfidfVectorizer()
matriz_campos = vectorizador_campos.fit_transform(textos_campos)

# Calculamos similitudes
similitudes_campos = cosine_similarity(matriz_campos)

print("SIMILITUD ENTRE TEXTOS DE CAMPOS SEMÁNTICOS RELACIONADOS")
print("=" * 60)

# Comparamos textos dentro del campo fútbol
print("\nCAMPO SEMÁNTICO: FÚTBOL")
print(f"Explícito vs Implícito 1: {similitudes_campos[0, 1]:.3f}")
print(f"Explícito vs Implícito 2: {similitudes_campos[0, 2]:.3f}")
print(f"Implícito 1 vs Implícito 2: {similitudes_campos[1, 2]:.3f}")

# Comparamos textos dentro del campo escritura
print("\nCAMPO SEMÁNTICO: ESCRITURA")
print(f"Explícito vs Implícito 1: {similitudes_campos[3, 4]:.3f}")
print(f"Explícito vs Implícito 2: {similitudes_campos[3, 5]:.3f}")
print(f"Implícito 1 vs Implícito 2: {similitudes_campos[4, 5]:.3f}")

# Comparación cruzada
print("\nCOMPARACIÓN ENTRE CAMPOS (debería ser baja):")
print(f"Fútbol explícito vs Escritura explícita: {similitudes_campos[0, 3]:.3f}")

print("\nANÁLISIS:")
print("- Las similitudes dentro de cada campo son bajas porque no comparten palabras exactas.")
print("- TF-IDF no reconoce que 'hinchada' y 'tribuna' se refieren al mismo concepto.")
print("- No entiende que 'relato', 'narración' y 'texto' están semánticamente relacionados.")


### Problema 4: La tragedia de los sinónimos

Este es quizás el problema más visible. Un buen escritor usa sinónimos para enriquecer su prosa. Pero para BoW/TF-IDF, "hermoso" y "bello" son palabras completamente diferentes, igual que "auto" y "coche" o "enojado" y "furioso".

Esto fragmenta artificialmente conceptos que deberían estar unidos.

In [ ]:
# --- Fragmentación de conceptos por sinónimos ---

textos_sinonimos = [
    "El paisaje era hermoso, me quedé contemplando la vista",
    "La vista era bella, no podía dejar de admirar el panorama",
    "Qué lindo lugar, la perspectiva desde aquí es preciosa",
    "Este sitio es precioso, la panorámica es realmente bonita"
]

# Análisis con TF-IDF
vectorizador_sin = TfidfVectorizer()
matriz_sin = vectorizador_sin.fit_transform(textos_sinonimos)

# Calculamos similitudes
similitudes_sin = cosine_similarity(matriz_sin)

print("ANÁLISIS DE SINÓNIMOS: CONCEPTOS FRAGMENTADOS")
print("=" * 50)

print("\nSIMILITUDES ENTRE DESCRIPCIONES:")
for i in range(len(textos_sinonimos)):
    for j in range(i + 1, len(textos_sinonimos)):
        sim = similitudes_sin[i, j]
        print(f"Descripción {i + 1} vs {j + 1}: {sim:.3f}")

print("\nTRAGEDIA DE LOS SINÓNIMOS:")
print("- Las 4 descripciones hablan del mismo concepto: un lugar hermoso con buena vista.")
print("- Pero TF-IDF las ve como textos diferentes porque usan sinónimos.")
print("- 'Hermoso', 'bello', 'lindo', 'precioso' expresan la misma idea.")
print("- 'Vista', 'panorama', 'perspectiva' se refieren al mismo elemento.")
print("- La riqueza léxica del español se convierte en una limitación técnica.")


---

## 4. Hacia la semántica computacional

### El salto conceptual necesario

Los problemas que acabamos de ver no son fallas de implementación que se puedan corregir con mejor preprocesamiento o stop words más sofisticadas. **Son limitaciones fundamentales de la representación bag-of-words.**

Para superarlas, necesitamos un cambio de paradigma: pasar de representaciones basadas en **co-ocurrencia de palabras exactas** a representaciones que capturen **significado semántico**.

### ¿Qué significa "semántica" en contexto computacional?

Cuando hablamos de semántica en PLN nos referimos a la capacidad de:

1. **Reconocer sinónimos**: "auto" y "coche" deben tener representaciones similares.
2. **Capturar relaciones conceptuales**: "médico" debe estar más cerca de "hospital" que de "computadora".
3. **Entender contexto**: distinguir "banco" (institución) de "banco" (asiento).
4. **Detectar analogías**: si "rey" es a "reina" como "hombre" es a "mujer".

### Vectores semánticos densos

La solución viene de representar palabras como **vectores densos** en un espacio multidimensional, donde:

- **Palabras similares** están **cerca** en el espacio vectorial.
- **Relaciones semánticas** se preservan como **relaciones geométricas**.
- **El contexto** determina la posición de cada palabra.
- **Las dimensiones** capturan aspectos abstractos del significado.

Experimentemos con esta idea usando spaCy, que incluye vectores pre-entrenados.

> **Nota importante sobre el modelo**: hasta ahora usamos `es_core_news_sm` (modelo chico), que no incluye vectores de palabras. Para los experimentos de esta sección necesitamos `es_core_news_md` (modelo mediano), que sí trae vectores pre-entrenados de 50 dimensiones. Si no lo tenés instalado, ejecutá en una terminal: `python -m spacy download es_core_news_md`.

In [ ]:
# --- Carga del modelo mediano de spaCy ---
# Necesitamos el modelo mediano (md) porque incluye vectores de palabras.
# El modelo chico (sm) que usamos en cuadernos anteriores no los tiene.

import subprocess
import sys

try:
    import spacy
    # Intentamos cargar el modelo mediano
    nlp = spacy.load("es_core_news_md")
    print("Modelo es_core_news_md cargado correctamente.")
except OSError:
    print("Instalando modelo es_core_news_md...")
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "es_core_news_md"])
    import spacy
    nlp = spacy.load("es_core_news_md")
    print("Modelo instalado y cargado.")
except ImportError:
    print("Instalando spaCy y modelo...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "spacy"])
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "es_core_news_md"])
    import spacy
    nlp = spacy.load("es_core_news_md")
    print("spaCy instalado desde cero.")

# Información sobre el modelo
nombre_modelo = nlp.meta['name']
idioma_modelo = nlp.meta['lang']
dimensiones = nlp.meta['vectors']['width']
cantidad_vectores = nlp.meta['vectors']['keys']

print(f"\nModelo: {nombre_modelo}")
print(f"Idioma: {idioma_modelo}")
print(f"Dimensiones de vectores: {dimensiones}")
print(f"Palabras con vectores: {cantidad_vectores:,}")


### Experimento 1: Resolviendo la tragedia de los sinónimos

Probemos si los vectores semánticos de spaCy pueden reconocer que dos palabras son sinónimos, algo que BoW no podía hacer. Vamos a comparar las mismas parejas que antes daban similitud cero.

In [ ]:
# --- Experimento 1: similitud semántica entre sinónimos ---

print("EXPERIMENTO 1: RESOLVIENDO LA TRAGEDIA DE LOS SINÓNIMOS")
print("=" * 60)

# Parejas de sinónimos que BoW veía como palabras completamente diferentes
palabra1_lista = ['padre', 'hermoso', 'auto', 'enojado', 'casa']
palabra2_lista = ['papá', 'bello', 'coche', 'furioso', 'hogar']

print("\nSIMILITUDES SEMÁNTICAS CON SPACY:")

for i in range(len(palabra1_lista)):
    palabra1 = palabra1_lista[i]
    palabra2 = palabra2_lista[i]

    # Obtenemos los vectores de spaCy
    token1 = nlp(palabra1)
    token2 = nlp(palabra2)

    # Calculamos la similitud (spaCy usa coseno internamente)
    similitud = token1.similarity(token2)

    print(f"'{palabra1}' <-> '{palabra2}': {similitud:.3f}")

print("\nCOMPARACIÓN:")
print("- BoW/TF-IDF: similitud = 0.000 (palabras diferentes, punto).")
print("- Vectores semánticos: similitud > 0.5 (reconoce la relación).")
print("- Los vectores capturan el conocimiento de que estas palabras son sinónimos.")


### Experimento 2: Capturando relaciones conceptuales

Ahora veamos si los vectores semánticos pueden reconocer que palabras como "fútbol", "cancha" y "gol" pertenecen al mismo campo semántico, sin que nosotros se lo digamos explícitamente.

In [ ]:
print("EXPERIMENTO 2: CAPTURANDO RELACIONES CONCEPTUALES")
print("=" * 55)

# Campos semánticos para explorar
nombres_campos = ['Fútbol', 'Escritura', 'Familia']
palabras_por_campo = [
    ['fútbol', 'partido', 'hinchada', 'cancha', 'gol', 'pelota'],
    ['escribir', 'texto', 'relato', 'historia', 'narración', 'palabras'],
    ['padre', 'hija', 'familia', 'casa', 'hogar', 'amor']
]

print("\nSIMILITUDES DENTRO DE CAMPOS SEMÁNTICOS:")

for campo_idx in range(len(nombres_campos)):
    nombre_campo = nombres_campos[campo_idx]
    palabras = palabras_por_campo[campo_idx]

    print(f"\n--- {nombre_campo.upper()} ---")

    # Calculamos similitud promedio dentro del campo
    similitudes_internas = []

    for i in range(len(palabras) - 1):
        for j in range(i + 1, len(palabras)):
            p1 = palabras[i]
            p2 = palabras[j]
            token1 = nlp(p1)
            token2 = nlp(p2)
            sim = token1.similarity(token2)
            similitudes_internas.append(sim)
            print(f"  {p1} <-> {p2}: {sim:.3f}")

    promedio = np.mean(similitudes_internas)
    print(f"  Similitud promedio en {nombre_campo}: {promedio:.3f}")


### Experimento 3: Comparación entre campos semánticos

Si los vectores realmente capturan significado, las palabras de campos diferentes deberían tener similitud baja entre sí. Verifiquemos.

In [ ]:
print("EXPERIMENTO 3: COMPARACIÓN ENTRE CAMPOS SEMÁNTICOS")
print("=" * 52)

# Comparaciones cruzadas entre campos
palabras_a = ['fútbol', 'padre', 'escribir', 'hija', 'cancha']
palabras_b = ['computadora', 'gol', 'pelota', 'narración', 'internet']
descripciones = [
    'Deporte vs Tecnología',
    'Familia vs Deporte',
    'Arte vs Deporte',
    'Familia vs Arte',
    'Deporte vs Tecnología'
]

print("\nSIMILITUDES ENTRE CAMPOS DIFERENTES:")
for i in range(len(palabras_a)):
    p1 = palabras_a[i]
    p2 = palabras_b[i]
    desc = descripciones[i]
    token1 = nlp(p1)
    token2 = nlp(p2)
    sim = token1.similarity(token2)
    print(f"{desc}: '{p1}' <-> '{p2}' = {sim:.3f}")

print("\nANÁLISIS:")
print("- Palabras del mismo campo semántico: similitud alta (> 0.4).")
print("- Palabras de campos diferentes: similitud baja (< 0.3).")
print("- Los vectores organizan automáticamente el conocimiento semántico.")
print("- Esa estructura estaba invisible en BoW/TF-IDF.")


### Experimento 4: Re-análisis semántico de los textos del corpus

Ahora apliquemos los vectores semánticos a nuestros fragmentos del corpus. Comparemos los textos de paternidad entre sí y los de fútbol entre sí. Si los vectores capturan significado, las similitudes dentro de cada tema deberían ser altas.

In [ ]:
# --- Re-análisis con vectores semánticos ---

print("EXPERIMENTO 4: RE-ANÁLISIS SEMÁNTICO DE TEXTOS DEL CORPUS")
print("=" * 58)

# Recuperamos los textos
nombres_analisis = [
    'Paternidad temprana',
    'Paternidad madura',
    'Fútbol pasión',
    'Fútbol nostalgia'
]
claves_analisis = [
    'paternidad_temprana',
    'paternidad_madura',
    'futbol_pasion',
    'futbol_nostalgia'
]

# Procesamos cada texto con spaCy para obtener vectores de documento
docs_spacy = []
for i in range(len(claves_analisis)):
    clave = claves_analisis[i]
    texto = textos_casciari_inspirados[clave]
    doc = nlp(texto)
    docs_spacy.append(doc)

# Calculamos similitudes
print("\nSIMILITUDES SEMÁNTICAS ENTRE TEXTOS:")
for i in range(len(nombres_analisis)):
    for j in range(i + 1, len(nombres_analisis)):
        nombre1 = nombres_analisis[i]
        nombre2 = nombres_analisis[j]
        similitud = docs_spacy[i].similarity(docs_spacy[j])
        print(f"{nombre1} <-> {nombre2}: {similitud:.3f}")

print("\nCOMPARACIÓN BoW vs VECTORES SEMÁNTICOS:")
print("\nBoW/TF-IDF:")
print("- Textos de mismo tema pero diferente léxico: similitud cercana a 0.")
print("- No detectaba la continuidad temática.")
print("\nVectores semánticos:")
print("- Textos de paternidad: similitud alta.")
print("- Textos de fútbol: similitud alta.")
print("- Reconoce la coherencia temática a pesar del vocabulario diferente.")


### Comprendiendo los vectores densos

Lo que acabamos de experimentar representa un salto cualitativo fundamental. En lugar de representar palabras como posiciones en un vocabulario gigante (BoW), ahora las representamos como **puntos en un espacio semántico continuo**.

**Características de los vectores semánticos densos:**

| Propiedad | BoW/TF-IDF | Vectores densos |
|---|---|---|
| Dimensionalidad | 30.000+ (una por palabra) | 50-300 |
| Tipo de valores | Enteros o pesos | Números reales continuos |
| Densidad | Mayormente ceros | Información en todas las dimensiones |
| Relaciones | No captura | Proximidad = similitud semántica |

**¿De dónde vienen estos vectores?**

Los vectores de spaCy fueron entrenados analizando millones de textos en español. El modelo aprendió que:
- Palabras que aparecen en contextos similares tienen significados similares.
- "Padre" y "papá" aparecen rodeadas de palabras parecidas: "mi", "querido", "familia".
- "Fútbol" y "computadora" aparecen en contextos muy diferentes, por eso están lejos.

En las próximas clases vas a profundizar en **cómo** se entrenan estos vectores usando Word2Vec, FastText y GloVe.

---

## 5. Preparación conceptual para word embeddings

Los experimentos que hicimos con spaCy te dieron una **experiencia directa** con vectores semánticos. Pero seguramente generaron preguntas nuevas:

- **¿Cómo se entrenan estos vectores?** ¿Qué algoritmos convierten texto en números que capturan semántica?
- **¿Por qué funcionan?** ¿Cuál es el principio matemático?
- **¿Cómo se decide la dimensionalidad?** ¿Por qué 50 o 300 dimensiones y no 1000?
- **¿Qué limitaciones tienen?** Si resuelven problemas de BoW, ¿crean problemas nuevos?

### Conceptos clave que vas a explorar

#### 1. La hipótesis distribucional
> "Una palabra se caracteriza por las compañías que mantiene" - J.R. Firth (1957)

Esta idea simple es la base teórica de todos los word embeddings:
- Palabras que aparecen en contextos similares tienen significados similares.
- Un algoritmo puede aprender esta regularidad y asignar vectores similares.

#### 2. Word2Vec
Word2Vec (Mikolov et al., 2013) propuso dos arquitecturas:
- **CBOW**: predice una palabra dado su contexto.
- **Skip-gram**: predice el contexto dada una palabra.

#### 3. FastText
Extiende Word2Vec considerando **sub-palabras**. Puede generar vectores para palabras que nunca vio durante el entrenamiento. Especialmente útil para idiomas con morfología rica como el español.

#### 4. GloVe
Combina estadísticas de co-ocurrencia de todo el corpus con métodos de predicción.

In [ ]:
# --- Adelanto: problemas que resolverás con word embeddings ---

print("PREPARACIÓN PARA LAS PRÓXIMAS CLASES")
print("=" * 42)

# Problema 1: Palabras fuera de vocabulario (OOV)
print("\n1. PROBLEMA OOV (Out of Vocabulary):")
palabras_raras = ['azulgrana', 'cuervos', 'santafesino', 'orsaiense']

for i in range(len(palabras_raras)):
    palabra = palabras_raras[i]
    token = nlp(palabra)
    tiene_vector = token.has_vector
    if tiene_vector:
        print(f"  '{palabra}': tiene vector en spaCy")
    else:
        print(f"  '{palabra}': SIN vector en spaCy")

print("  -> FastText resuelve este problema usando sub-palabras.")

# Problema 2: Analogías
print("\n2. ANALOGÍAS (álgebra de palabras):")
print("  Pregunta: rey - hombre + mujer = ?")
print("  -> Esperamos obtener 'reina'.")
print("  -> Vas a implementar búsqueda por analogías con Word2Vec.")

# Problema 3: Similitud semántica
print("\n3. BÚSQUEDA POR SIMILITUD:")
print("  Pregunta: palabras más similares a 'fútbol'.")
print("  -> Vas a implementar búsqueda de vecinos más cercanos.")
print("  -> Vas a comparar resultados entre Word2Vec, FastText y GloVe.")


### Reflexión crítica: limitaciones de los vectores semánticos

Antes de cerrar, es importante señalar que los vectores semánticos no son una solución perfecta. Resuelven problemas de BoW, pero traen limitaciones propias.

In [ ]:
# --- Limitaciones de los vectores semánticos ---

print("REFLEXIÓN CRÍTICA: LIMITACIONES DE VECTORES SEMÁNTICOS")
print("=" * 58)

# Polisemia
print("\n1. POLISEMIA (múltiples significados):")
palabras_polisemicas = ['banco', 'capital', 'carta', 'tiempo']
for i in range(len(palabras_polisemicas)):
    palabra = palabras_polisemicas[i]
    print(f"  '{palabra}': un solo vector para todos sus significados.")
print("  -> Los embeddings estáticos promedian todos los usos.")

# Sesgos
print("\n2. SESGOS SOCIALES EN LOS DATOS:")
p1_lista = ['doctor', 'programador']
p2_lista = ['enfermera', 'secretaria']
cat_lista = ['Profesiones de salud', 'Profesiones técnicas']
for i in range(len(p1_lista)):
    t1 = nlp(p1_lista[i])
    t2 = nlp(p2_lista[i])
    sim = t1.similarity(t2)
    print(f"  {cat_lista[i]}: '{p1_lista[i]}' <-> '{p2_lista[i]}' = {sim:.3f}")
print("  -> Los vectores reflejan sesgos presentes en los datos de entrenamiento.")

# Estaticidad
print("\n3. ESTATICIDAD:")
print("  - Un vector por palabra, independiente del contexto.")
print("  - No capturan cambios de significado en diferentes oraciones.")
print("  - Limitación que motivó el desarrollo de modelos contextuales (BERT, etc.).")

print("\nPERSPECTIVA:")
print("Los word embeddings fueron un avance revolucionario, pero no la solución final.")
print("En cursos avanzados vas a explorar embeddings contextuales que superan estas limitaciones.")


---

## 6. Síntesis y cierre

### El camino recorrido hoy

En este cuaderno transitamos desde las limitaciones concretas de BoW/TF-IDF hasta las posibilidades de las representaciones semánticas:

1. **Identificamos limitaciones específicas**: ceguera semántica, polisemia, fragmentación por sinónimos, incapacidad de capturar campos conceptuales.
2. **Experimentamos con soluciones**: vectores densos, similitud semántica, relaciones conceptuales.
3. **Desarrollamos intuición**: comprendimos por qué la geometría puede capturar significado.
4. **Preparamos conceptos**: establecimos las bases para word embeddings.

### Cambio de paradigma fundamental

| Aspecto | BoW/TF-IDF | Embeddings |
|---|---|---|
| Palabras como... | Símbolos discretos e independientes | Puntos en espacio semántico continuo |
| Representación basada en... | Co-ocurrencia exacta | Contexto distribucional |
| Espacio | Alta dimensionalidad, disperso | Dimensionalidad moderada, denso |
| Significado = | Frecuencia | Posición relativa |

### Conexión con las próximas clases

Más adelante vas a:
- **Implementar** los algoritmos que generan estos vectores.
- **Entrenar** modelos Word2Vec en español.
- **Comparar** FastText y GloVe en tareas específicas.
- **Evaluar** calidad de embeddings cuantitativamente.
- **Aplicar** vectores a problemas de similitud y analogías.

### Actividad breve

Antes de la próxima clase, reflexioná sobre estas preguntas:

1. **Conceptual:**
   - ¿Por qué BoW no puede distinguir sinónimos?
   - ¿Qué significa que dos vectores estén "cerca" en el espacio semántico?
   - ¿Por qué 300 dimensiones pueden capturar más información que 30.000?

2. **Práctico:**
   - ¿Cómo usarías similitud semántica para mejorar búsquedas en documentos?
   - ¿Qué ventajas tendría FastText vs Word2Vec para analizar textos argentinos?

3. **Crítico:**
   - ¿Qué sesgos podrían aparecer en embeddings entrenados con noticias?
   - ¿Cuándo seguirías usando BoW en lugar de embeddings?
   - ¿Qué limitaciones de embeddings estáticos motivaron BERT?

### Lo que viene después

Los conceptos que exploraste hoy son fundamentales para entender modelos de lenguaje modernos (GPT, BERT), sistemas de recomendación, traducción automática y análisis de sentimientos avanzado. **Los word embeddings fueron el primer paso hacia la IA que "entiende" texto.** En las próximas clases vas a aprender exactamente cómo funcionan por dentro.